<!-- paired-role-banner -->
> 🟩 **정답 노트북 10 · 참고 코드 창**
>
> 왼쪽 실습 노트북과 같은 셀 순서입니다. 결과만 복사하기보다 각 shape, 자료형, 손실값과
> 설계 이유를 먼저 예측한 뒤 필요한 부분만 확인하세요.

# 10. Tiny LM 학습·생성·KV cache

저장소의 작은 decoder-only Transformer를 실제 데이터에 학습시키고 checkpoint에서 복원한 뒤 생성 설정을 비교합니다. 모델 품질이 아니라 **학습/추론 파이프라인을 끝까지 연결하는 능력**이 목표입니다.

다루는 내용: vocabulary, next-token window, perplexity, checkpoint, temperature, top-k, top-p, greedy decoding, layer KV cache.

## 실행 전략

기본 설정은 CUDA → MPS → CPU 자동 선택으로 빠르게 실행됩니다.

- 합성 고객지원 티켓 일부만 사용
- `model_dim=32`, 1 layer, 2 epochs
- 결과는 `artifacts/notebook_10_tiny_lm_solution.pt`에 저장

tiny LM의 문장이 어색한 것은 정상입니다. 이번 실습에서는 loss, shape, 재현성, checkpoint 계약, 생성 비용을 관찰하세요.

<!-- api-explanations:start -->
<!-- api-explanations:keys=sys.path.insert -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>sys.path.insert(index, path)</code></summary>

#### `sys.path.insert(index, path)`

- **역할:** Python 모듈 검색 경로의 지정 위치에 경로를 추가합니다.
- **입력·반환:** index와 경로 문자열을 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 노트북에서 프로젝트 `src` 모듈을 import할 수 있게 합니다.
- **주의점:** 전역 import 상태를 바꾸므로 중복·우선순위와 실행 위치에 주의합니다.
- **공식 문서:** [sys.path.insert](https://docs.python.org/3/library/sys.html#sys.path)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
import sys
import time
from pathlib import Path

import pandas as pd
import torch
from torch.utils.data import DataLoader


def find_repo_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "src" / "llm_engineering_lab").is_dir():
            return candidate
    raise RuntimeError("저장소 루트 또는 notebooks/에서 실행하세요.")


ROOT = find_repo_root()
sys.path.insert(0, str(ROOT / "src"))
from llm_engineering_lab.acceleration import get_accelerator

ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
CHECKPOINT = ROOT / "artifacts" / "notebook_10_tiny_lm_solution.pt"
print(f"root={ROOT}\n{ACCELERATOR.summary()}\ncheckpoint={CHECKPOINT}")

In [ ]:
from llm_engineering_lab.torch_text import (
    Vocabulary,
    load_ticket_examples,
    seed_everything,
    split_examples,
)
from llm_engineering_lab.transformer import (
    LMTrainingConfig,
    LanguageModelDataset,
    TinyCausalTransformer,
    TinyTransformerConfig,
    build_lm_token_stream,
    fit_tiny_lm,
    load_tiny_lm_checkpoint,
)

seed_everything(42)

## 1) 데이터와 vocabulary

`customer_support_tickets.csv`의 텍스트를 group-safe split한 뒤 작은 부분집합을 사용합니다. `mini_corpus.txt`는 다른 도메인 corpus를 추가했을 때 vocabulary와 perplexity가 어떻게 달라지는지 비교하기 위한 확장 재료입니다.

In [ ]:
examples = load_ticket_examples(ROOT / "data" / "customer_support_tickets.csv")
train_examples, valid_examples = split_examples(examples, validation_ratio=0.2, seed=42)
train_examples = train_examples[:50]
valid_examples = valid_examples[:15]
mini_corpus = (ROOT / "data" / "raw" / "mini_corpus.txt").read_text(encoding="utf-8")

vocabulary = Vocabulary.build((item.text for item in train_examples), max_size=1200)
assert vocabulary.pad_id == 0
print(
    f"train={len(train_examples)} valid={len(valid_examples)} vocab={len(vocabulary)} "
    f"mini_corpus_chars={len(mini_corpus)}"
)
print(
    "special ids:",
    vocabulary.pad_id,
    vocabulary.unk_id,
    vocabulary.bos_id,
    vocabulary.eos_id,
)

In [ ]:
sample_text = train_examples[0].text
sample_ids = vocabulary.encode(sample_text, add_bos=True, add_eos=True)
decoded = vocabulary.decode(sample_ids)

assert sample_ids[0] == vocabulary.bos_id
assert sample_ids[-1] == vocabulary.eos_id
print("text:", sample_text)
print("token ids:", sample_ids[:24], "..." if len(sample_ids) > 24 else "")
print("decoded:", decoded)

## 2) packed token stream과 고정 길이 window

각 문서 앞뒤에 BOS/EOS를 붙여 하나의 token stream으로 연결하고, 겹치는 window를 만듭니다. 각 sample의 `input_ids`와 `labels`는 같지만 loss 내부에서 label이 한 칸 이동합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=DataLoader -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>DataLoader(dataset, batch_size=1, shuffle=False, ...)</code></summary>

#### `DataLoader(dataset, batch_size=1, shuffle=False, ...)`

- **역할:** dataset에서 mini-batch를 반복 생성합니다.
- **입력·반환:** dataset과 batching 설정을 받고 iterable loader를 반환합니다.
- **이 셀에서 쓰는 이유:** 학습 데이터를 batch 단위로 섞고 모델에 공급합니다.
- **주의점:** Windows 다중 worker는 진입점 보호가 필요하며 마지막 batch 크기도 확인합니다.
- **공식 문서:** [DataLoader](https://docs.pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
train_stream = build_lm_token_stream((item.text for item in train_examples), vocabulary)
valid_stream = build_lm_token_stream((item.text for item in valid_examples), vocabulary)
BLOCK_SIZE = 24
train_dataset = LanguageModelDataset(
    train_stream, block_size=BLOCK_SIZE, stride=BLOCK_SIZE, pad_id=vocabulary.pad_id
)
valid_dataset = LanguageModelDataset(
    valid_stream, block_size=BLOCK_SIZE, stride=BLOCK_SIZE, pad_id=vocabulary.pad_id
)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=0)
valid_loader = DataLoader(valid_dataset, batch_size=8, shuffle=False, num_workers=0)

batch = next(iter(train_loader))
assert batch["input_ids"].shape[1] == BLOCK_SIZE
assert (
    batch["input_ids"].shape == batch["attention_mask"].shape == batch["labels"].shape
)
print("stream lengths:", len(train_stream), len(valid_stream))
print("windows:", len(train_dataset), len(valid_dataset))
print({key: (tuple(value.shape), value.dtype) for key, value in batch.items()})

## 3) CPU용 tiny Transformer 구성

`context_length`는 학습 window 이상이며 생성 시에는 `prompt_tokens + max_new_tokens`보다 커야 합니다. `model_dim`은 `num_heads`로 나누어져야 합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=Module.parameters -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>model.parameters(recurse=True)</code></summary>

#### `model.parameters(recurse=True)`

- **역할:** 모듈에 등록된 학습 파라미터 iterator를 반환합니다.
- **입력·반환:** 하위 모듈 포함 여부를 받고 `Parameter` iterator를 반환합니다.
- **이 셀에서 쓰는 이유:** optimizer가 갱신할 weight와 bias를 전달합니다.
- **주의점:** iterator는 한 번 소비되며 buffer와 일반 tensor attribute는 포함하지 않습니다.
- **공식 문서:** [Module.parameters](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
model_config = TinyTransformerConfig(
    vocab_size=len(vocabulary),
    pad_id=vocabulary.pad_id,
    max_seq_length=48,
    model_dim=32,
    num_heads=4,
    num_layers=1,
    feed_forward_dim=64,
    dropout=0.1,
)
model = TinyCausalTransformer(model_config).to(DEVICE)
parameter_count = sum(parameter.numel() for parameter in model.parameters())

assert model_config.model_dim % model_config.num_heads == 0
print(model_config)
print(
    f"parameters={parameter_count:,} "
    f"({parameter_count * 4 / 1024:.1f} KiB in fp32, optimizer 제외)"
)

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.isfinite,Tensor.item -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 2개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.isfinite(input)</code></summary>

#### `torch.isfinite(input)`

- **역할:** 텐서의 각 원소가 유한수인지 검사합니다.
- **입력·반환:** 텐서를 받고 같은 shape의 불리언 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** loss나 gradient의 NaN·무한대 발생을 빠르게 찾습니다.
- **주의점:** 검사 결과는 미분 대상이 아니며 원인 자체를 고치지는 않습니다.
- **공식 문서:** [torch.isfinite](https://docs.pytorch.org/docs/stable/generated/torch.isfinite.html)

</details>

<details>
<summary><code>tensor.item()</code></summary>

#### `tensor.item()`

- **역할:** 원소 하나짜리 텐서에서 Python 숫자를 꺼냅니다.
- **입력·반환:** 단일 원소 텐서를 받고 Python 숫자를 반환합니다.
- **이 셀에서 쓰는 이유:** loss나 metric을 출력·기록 가능한 값으로 바꿉니다.
- **주의점:** 여러 원소 텐서에는 쓸 수 없고 GPU에서는 동기화를 일으킬 수 있습니다.
- **공식 문서:** [Tensor.item](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.item.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
smoke = {key: value[:2].to(DEVICE) for key, value in batch.items()}
output = model(
    smoke["input_ids"],
    attention_mask=smoke["attention_mask"],
    labels=smoke["labels"],
)
assert output.logits.shape == (2, BLOCK_SIZE, len(vocabulary))
assert output.loss is not None and torch.isfinite(output.loss)
print(
    "logits:", tuple(output.logits.shape), "initial loss:", round(output.loss.item(), 4)
)

## 4) 학습과 best checkpoint

`fit_tiny_lm`은 AdamW로 학습하고 validation loss가 가장 낮은 epoch의 model/optimizer/config/metric/metadata를 저장합니다. 반복 실행하면 이 노트북 전용 checkpoint를 덮어씁니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=perf_counter -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>perf_counter()</code></summary>

#### `perf_counter()`

- **역할:** 짧은 구간 측정에 적합한 고해상도 단조 시계를 읽습니다.
- **입력·반환:** 인자 없이 호출하고 초 단위 부동소수점 값을 반환합니다.
- **이 셀에서 쓰는 이유:** 학습·추론 latency의 시작과 끝 차이를 측정합니다.
- **주의점:** 절댓값에는 의미가 없고 GPU 시간은 실행 동기화 뒤 측정해야 합니다.
- **공식 문서:** [perf_counter](https://docs.python.org/3/library/time.html#time.perf_counter)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
training_config = LMTrainingConfig(
    epochs=2,
    learning_rate=2e-3,
    weight_decay=0.01,
    gradient_clip_norm=1.0,
)
ACCELERATOR.synchronize()
started = time.perf_counter()
history = fit_tiny_lm(
    model,
    train_loader,
    validation_loader=valid_loader,
    config=training_config,
    device=DEVICE,
    checkpoint_path=CHECKPOINT,
    checkpoint_extra={
        "vocabulary": vocabulary.to_dict(),
        "source": "data/customer_support_tickets.csv",
        "notebook": "10_tiny_lm_training_and_generation.ipynb",
        "seed": 42,
    },
)
ACCELERATOR.synchronize()
elapsed = time.perf_counter() - started
assert CHECKPOINT.is_file()
print(f"training elapsed={elapsed:.2f}s, checkpoint={CHECKPOINT.relative_to(ROOT)}")

<!-- api-explanations:start -->
<!-- api-explanations:keys=pd.DataFrame -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>pd.DataFrame(data=None, index=None, columns=None, ...)</code></summary>

#### `pd.DataFrame(data=None, index=None, columns=None, ...)`

- **역할:** 행·열 구조의 DataFrame을 만듭니다.
- **입력·반환:** 표 데이터와 열·그룹 설정을 받고 DataFrame, Series 또는 집계 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** 학습 전 데이터 분포·결측·중복과 분할 기준을 점검합니다.
- **주의점:** dtype 추론, 결측값, index 보존과 grouping 후 shape 변화를 확인해야 합니다.
- **공식 문서:** [pd.DataFrame](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
history_frame = pd.DataFrame(
    {
        "epoch": record["epoch"],
        "train_loss": record["train"]["loss"],
        "train_ppl": record["train"]["perplexity"],
        "valid_loss": record["validation"]["loss"],
        "valid_ppl": record["validation"]["perplexity"],
    }
    for record in history
)
print(history_frame.round(3).to_string(index=False))
print(
    "관찰: tiny dataset에서는 validation metric 변동이 크므로 단일 숫자를 과신하지 마세요."
)

## 5) 새 프로세스처럼 checkpoint 복원

가중치만 저장하면 vocabulary id나 model dimension을 잃기 쉽습니다. metadata와 config를 함께 복원하고 CPU `map_location`을 명시합니다. pickle 기반 checkpoint는 신뢰할 수 있는 로컬 파일만 읽으세요.

<!-- api-explanations:start -->
<!-- api-explanations:keys=Tensor.to -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>tensor.to(device=None, dtype=None)</code></summary>

#### `tensor.to(device=None, dtype=None)`

- **역할:** 텐서를 지정 장치나 dtype으로 변환합니다.
- **입력·반환:** 장치·dtype을 받고 변환된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 모델과 입력의 장치·정밀도를 일치시킵니다.
- **주의점:** 원본과 조건이 같으면 복사 없이 같은 텐서를 돌려줄 수 있습니다.
- **공식 문서:** [Tensor.to](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.to.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
restored_model, payload = load_tiny_lm_checkpoint(CHECKPOINT, map_location=DEVICE)
restored_model.to(DEVICE).eval()
restored_vocabulary = Vocabulary.from_dict(payload["extra"]["vocabulary"])

assert restored_model.config == model_config
assert len(restored_vocabulary) == len(vocabulary)
print("saved epoch:", payload["epoch"])
print("saved metrics:", payload["metrics"])
print("metadata:", payload["extra"])

## 6) 생성 helper

`temperature=0`은 greedy decoding입니다. 0보다 크면 확률 sampling을 하고 top-k/top-p로 후보를 제한합니다. 고정 seed는 실험 비교에 유용하지만 실제 서비스의 다양성 정책과는 별개입니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.tensor,torch.manual_seed -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 2개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.tensor(data, dtype=None, device=None)</code></summary>

#### `torch.tensor(data, dtype=None, device=None)`

- **역할:** 데이터를 복사해 새 PyTorch 텐서를 만듭니다.
- **입력·반환:** 배열형 데이터와 선택적 dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 상수·label·축소 실험 입력을 명시적으로 구성합니다.
- **주의점:** 기존 텐서에서 호출하면 복사가 생기므로 경우에 따라 `as_tensor`가 낫습니다.
- **공식 문서:** [torch.tensor](https://docs.pytorch.org/docs/stable/generated/torch.tensor.html)

</details>

<details>
<summary><code>torch.manual_seed(seed)</code></summary>

#### `torch.manual_seed(seed)`

- **역할:** PyTorch 난수 생성기의 시드를 설정합니다.
- **입력·반환:** 정수 시드를 받고 `Generator`를 반환합니다.
- **이 셀에서 쓰는 이유:** 가중치 초기화와 텐서 샘플링을 반복 가능하게 만듭니다.
- **주의점:** 완전한 결정성에는 알고리즘·장치 설정도 추가로 필요할 수 있습니다.
- **공식 문서:** [torch.manual_seed](https://docs.pytorch.org/docs/stable/generated/torch.manual_seed.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
def complete(
    prompt: str,
    *,
    max_new_tokens: int = 10,
    temperature: float = 0.0,
    top_k: int | None = None,
    top_p: float = 1.0,
    seed: int = 123,
    use_cache: bool = True,
) -> tuple[str, torch.Tensor]:
    prompt_limit = restored_model.config.max_seq_length - max_new_tokens
    ids = restored_vocabulary.encode(prompt, add_bos=True, max_length=prompt_limit)
    input_tensor = torch.tensor([ids], dtype=torch.long, device=DEVICE)
    if DEVICE.type in {"cpu", "cuda"}:
        generator = torch.Generator(device=DEVICE.type).manual_seed(seed)
    else:
        torch.manual_seed(seed)
        generator = None
    generated = restored_model.generate(
        input_tensor,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_k=top_k,
        top_p=top_p,
        eos_id=restored_vocabulary.eos_id,
        use_cache=use_cache,
        generator=generator,
    )
    return restored_vocabulary.decode(generated[0].tolist()), generated

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.equal -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.equal(input, other)</code></summary>

#### `torch.equal(input, other)`

- **역할:** 두 텐서의 크기와 원소가 정확히 같은지 검사합니다.
- **입력·반환:** 두 텐서를 받고 Python `bool` 하나를 반환합니다.
- **이 셀에서 쓰는 이유:** 토큰 ID나 정수 마스크처럼 완전 일치가 필요한 결과를 확인합니다.
- **주의점:** dtype이 달라도 값이 같으면 참일 수 있고 NaN은 자기 자신과 같지 않습니다.
- **공식 문서:** [torch.equal](https://docs.pytorch.org/docs/stable/generated/torch.equal.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
prompt = "결제가 두 번 처리되어 환불을 요청합니다"
greedy_text, greedy_ids = complete(prompt, temperature=0.0, max_new_tokens=10)
greedy_text_2, greedy_ids_2 = complete(
    prompt, temperature=0.0, max_new_tokens=10, seed=999
)

assert torch.equal(greedy_ids, greedy_ids_2)  # greedy는 seed와 무관
print("greedy:", greedy_text)
print("shape:", tuple(greedy_ids.shape))

## 7) temperature 비교

낮은 temperature는 높은 logit의 차이를 확대하고, 높은 temperature는 분포를 평평하게 합니다. 작은 모델에서는 높은 값이 문맥을 급격히 무너뜨릴 수 있습니다.

In [ ]:
for temperature in (0.4, 0.8, 1.3):
    text, _ = complete(
        prompt,
        temperature=temperature,
        top_k=None,
        top_p=1.0,
        seed=2026,
    )
    print(f"temperature={temperature}: {text}")

## 8) top-k와 top-p(nucleus) 비교

- top-k: 확률이 높은 고정 개수 후보만 유지
- top-p: 누적 확률이 `p`가 될 때까지 동적 개수 후보 유지

둘을 함께 쓰면 두 조건을 모두 통과한 후보에서 sampling합니다.

In [ ]:
settings = [
    {"name": "unfiltered", "top_k": None, "top_p": 1.0},
    {"name": "top-k=5", "top_k": 5, "top_p": 1.0},
    {"name": "top-p=0.7", "top_k": None, "top_p": 0.7},
    {"name": "k=10,p=0.9", "top_k": 10, "top_p": 0.9},
]
for setting in settings:
    text, _ = complete(
        prompt,
        temperature=0.8,
        top_k=setting["top_k"],
        top_p=setting["top_p"],
        seed=77,
    )
    print(f"{setting['name']:>12}: {text}")

## 9) KV cache shape와 greedy 동등성

첫 forward에서 각 layer의 K/V는 `[B,H,T,Dh]`로 저장됩니다. 다음 token에서는 이전 `T`개를 다시 계산하지 않고 새 query 하나만 처리합니다. cache는 계산을 줄이지만 layer×context만큼 메모리를 사용합니다.

In [ ]:
probe_ids = torch.tensor(
    [restored_vocabulary.encode(prompt, add_bos=True, max_length=20)],
    dtype=torch.long,
    device=DEVICE,
)
probe_mask = probe_ids.ne(restored_vocabulary.pad_id)
cache_output = restored_model(probe_ids, attention_mask=probe_mask, use_cache=True)
caches = cache_output.past_key_values

assert caches is not None and len(caches) == restored_model.config.num_layers
first_key, first_value = caches[0]
expected_head_dim = restored_model.config.model_dim // restored_model.config.num_heads
assert first_key.shape == first_value.shape
assert first_key.shape == (
    1,
    restored_model.config.num_heads,
    probe_ids.shape[1],
    expected_head_dim,
)
cache_bytes = (
    sum(cache.key.numel() + cache.value.numel() for cache in caches)
    * first_key.element_size()
)
print("layer-0 K/V shape:", tuple(first_key.shape))
print(f"current cache~{cache_bytes / 1024:.2f} KiB")

In [ ]:
def greedy_benchmark(use_cache: bool) -> tuple[torch.Tensor, float]:
    ACCELERATOR.synchronize()
    started = time.perf_counter()
    _, ids = complete(
        prompt,
        max_new_tokens=8,
        temperature=0.0,
        top_k=None,
        top_p=1.0,
        use_cache=use_cache,
    )
    ACCELERATOR.synchronize()
    return ids, time.perf_counter() - started


ids_without_cache, no_cache_seconds = greedy_benchmark(False)
ids_with_cache, cache_seconds = greedy_benchmark(True)
assert torch.equal(ids_without_cache, ids_with_cache)
print(f"no-cache={no_cache_seconds * 1000:.2f}ms cache={cache_seconds * 1000:.2f}ms")
print("작은 모델/짧은 문장에서는 cache 관리 오버헤드가 더 클 수도 있습니다.")

## 체크포인트 운영 체크리스트

checkpoint에는 최소한 다음이 필요합니다.

- model state와 model config
- tokenizer/vocabulary 버전
- 학습 데이터 출처와 split/seed
- optimizer state와 완료 epoch
- validation metric, 코드/git revision

`torch.load`/pickle 계열 파일은 임의 사용자가 업로드한 것을 로드하지 마세요. 실제 배포에서는 artifact hash, 서명, 허용 경로도 검증합니다.

## TODO와 완료 기준

- **TODO 10-A:** `epochs=1/3/8`에서 train/validation perplexity와 시간을 표로 비교하세요.
- **TODO 10-B:** `block_size`와 `stride`를 바꾸고 전체 target token 수가 어떻게 달라지는지 확인하세요.
- **TODO 10-C:** 같은 prompt를 seed 5개로 생성해 고유 token 비율과 반복률을 계산하세요.
- **TODO 10-D:** KV cache byte 수를 layer/context/dtype별 함수로 일반화하세요.
- **TODO 10-E:** `mini_corpus.txt`를 학습 corpus에 추가했을 때 support-ticket validation perplexity가 좋아지는지 검증하세요.
- **TODO 10-F (현업):** checkpoint metadata에 git commit hash와 데이터 SHA-256을 추가하세요.

완료 기준: checkpoint에서 새 모델과 vocabulary를 복원하고, 생성 설정과 cache의 품질/비용 trade-off를 설명할 수 있습니다.